In [ ]:
# ==============================
# Importación de librerías
# ==============================
import cv2              # Librería OpenCV para visión por computadora
import mediapipe as mp  # Librería MediaPipe para detección de manos
import numpy as np      # Librería NumPy para cálculos numéricos
import pyautogui        # Librería para controlar el mouse y teclado
import time
import keyboard
import scipy.signal as sig
# ==============================
# Configuración de MediaPipe y cámara
# ==============================
mp_hands = mp.solutions.hands            # Modelo de detección de manos de MediaPipe
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW) # Abre la cámara (índice 0), CAP_DSHOW es para Windows

window_size = 1000  # Cuántos frames promediar
positions = []   # Lista circular de posiciones previas

Fs = 100000
fc = 4000  # Frecuencia de corte del filtro
orden = 1  # Orden del filtro
b, a = sig.butter(orden, fc/(0.5*Fs), btype='low', analog=False, output='ba')



def suavizar_pos(x, y):
    positions.append((x, y))
    if len(positions) > window_size:
        positions.pop(0)


    #descomprime la posicion almacenada(x,y) en 2 variables
    xs, ys = zip(*positions)

    #FIltra la secuencia entera en la ventana
    x0_filtered = sig.lfilter(b, a, xs)
    y0_filtered = sig.lfilter(b, a, ys)

    #toma el ultimo elemento
    x_suavizado = x0_filtered[-1]
    y_suavizado = y0_filtered[-1]

    return x_suavizado, y_suavizado


# ==============================
# SUB BLOQUES - funciones principales
# ==============================
def mouse_ubicacion(coord_mano):
    mouse_x = mouse_y = 100
    # ==============================
    # Si se detectan manos
    # ==============================
    if coord_mano.multi_hand_landmarks is not None:
        for hand_landmarks in coord_mano.multi_hand_landmarks:
            mouse_x = int(hand_landmarks.landmark[0].x * width_frame) #libreria py autogui tamaño pantalla
            mouse_y = int(hand_landmarks.landmark[0].y * height_frame)
            mouse_x,mouse_y =    suavizar_pos(mouse_x, mouse_y)

    return mouse_x , mouse_y



def mouse_gesto_operacion(coord_mano):
    BI=BD=0
    if coord_mano.multi_hand_landmarks:
        for hand_landmarks in coord_mano.multi_hand_landmarks:
            # Ejemplo: dedo índice extendido => click izquierdo
            BD  = hand_landmarks.landmark[8].y > hand_landmarks.landmark[6].y
            BI  = hand_landmarks.landmark[12].y > hand_landmarks.landmark[10].y
    return BD,BI

# ==============================
# BLOQUES - funciones principales
# ==============================
def Deteccion_mano_openCV(frame):
    frame = cv2.flip(frame, 1)      # Voltear la imagen horizontalmente (como espejo)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) #lo necesita?
    coord_mano = hands.process(frame_rgb) # Procesar frame con MediaPipe para detectar manos
    return coord_mano

def Deteccion_gestos(coord_mano):
    mouse_x , mouse_y = mouse_ubicacion(coord_mano)
    BD, BI = mouse_gesto_operacion(coord_mano)

    return BD , BI , mouse_x , mouse_y


def f_tecla(FF,boton,tecla):
# ===== Control botón derecho (usa FFD) =====
    if FF == 0 and boton == 1:
        pyautogui.mouseDown(button=tecla)
        FF = 1                       # Modifica el estado global FFD

    if FF == 1 and boton == 0:
        pyautogui.mouseUp(button=tecla)
        FF = 0                       # Modifica el estado global FFD
    # ===== Fin control botón derecho =====
    return FF

FFD = FFI = 0
def mouse_virtual(x_frame, y_frame , boton_izq, boton_der):
    global FFD,FFI
    width_pantalla,height_pantalla = pyautogui.size()

    x = (int)((x_frame - 1/2*width_frame)*2.2*width_pantalla/width_frame)
    y = (int)((y_frame - 1/2*height_frame)*2.2*height_pantalla/height_frame)

    # ... (Control de movimiento del cursor)
    if (x > 0) and (y > 0):
        try:
            pyautogui.moveTo(x, y)
        except pyautogui.FailSafeException:
            print("Movimiento detenido: mouse llegó a la esquina")
    else:
        pyautogui.moveTo(5, 5)

    FFD = f_tecla(FFD,boton_der,'right')
    FFI = f_tecla(FFI,boton_izq,'left')


# ==============================
# Bucle principal con MediaPipe Hands
# ==============================
with mp_hands.Hands(
    static_image_mode=False,        # Procesa video en vivo
    max_num_hands=1,                # Máximo 1 mano detectada
    min_detection_confidence=0.7) as hands:  # Confianza mínima para detección

    while True:
        ret, frame = cap.read()     # Leer un frame de la cámara
        if ret == False:            # Si no hay frame, salir
            break

        height_frame, width_frame, _ = frame.shape   # Obtener dimensiones del frame

        coord_mano = Deteccion_mano_openCV(frame)   # Procesar frame con MediaPipe para detectar manos

        BD, BI,mouse_x , mouse_y = Deteccion_gestos(coord_mano)

        mouse_virtual(mouse_x ,mouse_y , BD, BI)

        if keyboard.is_pressed("esc"):
            print("Saliste del bucle.")
            break
        time.sleep(0.05)
# ==============================
# Liberar recursos
# ==============================
cap.release()           # Liberar la cámara
cv2.destroyAllWindows() # Cerrar todas las ventanas de OpenCV
# barra que modifique la frecuencia de corte
